# HW3 report runs (Colab)

Runs every measurement the HW3 report needs in **one** Colab session, saves each program's
real stdout to `logs/*.txt`, renders each log to `shots/*.png` (an image of that exact output),
runs the Triton notebook end-to-end, regenerates the Part B plots and `results.json`, and
zips it all as `hw3_report_outputs.zip`.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.
The Part B sweep (scenario 1 at K=10 million) takes several minutes; total is about 10-15 min.

In [ ]:
REPO = "https://github.com/hardik5902/HPML.git"   # private repo? use https://<TOKEN>@github.com/...
!nvidia-smi
!rm -rf /content/HPML && git clone -q $REPO /content/HPML
%cd /content/HPML/HPML_HW3_starter
!pip -q install jsonschema nbconvert nbclient

In [ ]:
import subprocess, pathlib, json, shutil
OUT = pathlib.Path("/content/report_out")
shutil.rmtree(OUT, ignore_errors=True)
for d in ("logs", "shots", "plots"):
    (OUT / d).mkdir(parents=True, exist_ok=True)
ROOT = pathlib.Path("/content/HPML/HPML_HW3_starter")

def run(name, cmd, cwd=ROOT):
    """Run a shell command, save '$ cmd' + its real stdout/stderr to logs/<name>.txt."""
    p = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    txt = f"$ {cmd}\n" + p.stdout + p.stderr
    (OUT / "logs" / f"{name}.txt").write_text(txt)
    print(txt)
    if p.returncode != 0:
        print(f"!!! {name} exited with code {p.returncode}")
    return txt

cc = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                    shell=True, capture_output=True, text=True).stdout.strip().splitlines()[0]
ARCH = "sm_" + cc.replace(".", "")
print("Detected ARCH =", ARCH)

## Setup info and build

In [ ]:
run("00_setup",
    "nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv; "
    "nvcc --version | tail -n 2; "
    "lscpu | grep -E '^Model name|^CPU\\(s\\)'; "
    "grep PRETTY_NAME /etc/os-release; uname -r; "
    "free -g | head -n 2")
run("01_build", f"cd src && make clean > /dev/null && make ARCH={ARCH} 2>&1 | tail -n 3")

## Part A

In [ ]:
A = ROOT / "src/parta"
run("A_Q1_vecadd00", "./vecadd00 500 && ./vecadd00 1000 && ./vecadd00 2000", A)
run("A_Q2_vecadd01", "./vecadd01 500 && ./vecadd01 1000 && ./vecadd01 2000", A)
# matmult side = arg * FOOTPRINT_SIZE: 16*{16,32,64} and 32*{8,16,32} both give 256, 512, 1024
run("A_Q3_matmult00", "./matmult00 16 && ./matmult00 32 && ./matmult00 64", A)
run("A_Q4_matmult01", "./matmult01 8 && ./matmult01 16 && ./matmult01 32", A)

## Part B (all K, all three scenarios)

In [ ]:
B = ROOT / "src/partb"
run("B_Q1_cpu", "./q1_cpu", B)
run("B_Q2_malloc", "./q2_malloc", B)
run("B_Q3_unified", "./q3_unified", B)

In [ ]:
run("B_Q4_plot", f"python src/partb/q4_plot.py --bindir src/partb --out-dir {OUT/'plots'}")

## Part C (default N=2048, d=64, plus N=1024 and N=4096 for the scaling discussion)

In [ ]:
C = ROOT / "src/partc"
run("C1_naive", "./c1_naive", C)
run("C2_tiled", "./c2_tiled", C)
run("C3_cublas", "./c3_cublas", C)
run("C_sizes", " && ".join(f"./{p} {n} 64" for n in (1024, 4096) for p in ("c1_naive", "c2_tiled", "c3_cublas")), C)

## Part C4: execute the Triton notebook (outputs saved into it)

In [ ]:
run("C4_nbconvert",
    "jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=900 "
    f"src/partc/c4_triton.ipynb --output {OUT/'c4_triton.ipynb'}")
nb = json.loads((OUT / "c4_triton.ipynb").read_text())
lines = ["$ (outputs of the executed src/partc/c4_triton.ipynb)"]
for c in nb["cells"]:
    for o in c.get("outputs", []):
        if o.get("output_type") == "stream":
            lines.append("".join(o["text"]).rstrip("\n"))
        elif o.get("output_type") == "error":
            lines.append(o["ename"] + ": " + o["evalue"])
txt = "\n".join(lines) + "\n"
(OUT / "logs" / "C4_triton.txt").write_text(txt)
print(txt)

## results.json from the same session

In [ ]:
run("Z_grade_local", f"python grade_local.py --arch {ARCH} --out {OUT/'results.json'} 2>&1 | tail -n 5")
r = json.loads((OUT / "results.json").read_text())
print("mode:", r["meta"]["mode"])
for part in ("part_a", "part_b", "part_c"):
    for k, v in r[part].items():
        if isinstance(v, dict) and "status" in v:
            print(f"  {part}.{k}: {v['status']}")
print("  part_c.correctness:", r["part_c"]["correctness"])

## Render each log as an image, zip, download

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def render(log_path, png_path, fontsize=8.5):
    txt = log_path.read_text().rstrip("\n").expandtabs(4)
    lines = txt.splitlines() or [""]
    cw, lh = fontsize * 0.62 / 72, fontsize * 1.25 / 72          # char width / line height (in)
    W = max(60, max(len(l) for l in lines)) * cw + 0.4
    H = len(lines) * lh + 0.4
    fig = plt.figure(figsize=(W, H), dpi=160)
    fig.patch.set_facecolor("#1e1e1e")
    fig.text(0.2 / W, 1 - 0.2 / H, txt, family="DejaVu Sans Mono", fontsize=fontsize,
             color="#e8e8e8", va="top", ha="left", linespacing=1.25)
    fig.savefig(png_path, facecolor=fig.get_facecolor())
    plt.close(fig)

for log in sorted((OUT / "logs").glob("*.txt")):
    render(log, OUT / "shots" / (log.stem + ".png"))
print(sorted(p.name for p in (OUT / "shots").iterdir()))

!cd /content && rm -f hw3_report_outputs.zip && zip -qr hw3_report_outputs.zip report_out
from google.colab import files
files.download("/content/hw3_report_outputs.zip")